In [2]:
!pip install -q google-genai pydantic pandas

In [3]:
import getpass
from google import genai

# Get a free key at https://aistudio.google.com/apikey  (never put the key in GitHub)
API_KEY = ""          # leave empty to be asked; or paste it here ONLY while running, delete before sharing
if not API_KEY:
    API_KEY = getpass.getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)
print("Gemini client ready")


Paste your Gemini API key: ··········
Gemini client ready


In [4]:
import os, glob, json, time, getpass
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

# Upload follow_up_notes, clinic, appointment CSVs (skipped if already in /content)
def find(keyword):
    hits = [f for f in glob.glob("/content/*.csv") if keyword in f.lower()]
    return hits[0] if hits else None

if not (find("note") and find("appointment")):
    from google.colab import files
    files.upload()          # select all 3 CSV files

NOTES_FILE = find("note")
APPT_FILE  = find("appointment")
print("notes file       :", NOTES_FILE)
print("appointment file :", APPT_FILE)

# ---------- settings ----------
MODEL_NAME = "gemini-3-flash-preview"   # <- change to the Flash model from Session 22.1 if different
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

SCHEMA = {   # root_cause is a plain string (NOT an enum); allowed values enforced afterwards
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = ("You are a careful analyst who reads call-centre notes from outpatient clinics in India "
               "and labels why a patient missed an appointment. Notes may be in English, Hindi or Hinglish. "
               "Never invent facts that are not in the note.")

API_CALLS = {"4.1": 0, "4.2": 0, "4.3": 0}   # counts every real API call

Saving follow up notes.csv.csv to follow up notes.csv.csv
Saving clinic.csv.csv to clinic.csv.csv
Saving appointment.csv.csv to appointment.csv.csv
notes file       : /content/follow up notes.csv.csv
appointment file : /content/appointment.csv.csv


In [5]:
def build_craft_prompt(note_text):
    """V2 prompt = CRAFT: Context, Role, Action, Tone, Format"""
    return f"""CONTEXT: A clinic patient missed an appointment. A call-centre agent wrote the note below after phoning the patient. The note may be in English, Hindi or Hinglish.
ROLE: You are a healthcare operations analyst.
ACTION: Identify the single main root cause of the missed appointment. root_cause MUST be exactly one of: {", ".join(ALLOWED)}. If the note does not clearly state a reason, answer "other" rather than guess. Give confidence between 0 and 1, a summary of at most 25 words, and one concrete recommended_action for the clinic.
TONE: Neutral, factual and concise.
FORMAT: Return only a JSON object with the keys root_cause, confidence, summary, recommended_action.

NOTE: {note_text}"""

def build_v1_prompt(note_text):
    return f"Classify this note: {note_text}"

def _generate(prompt, task):
    config_kwargs = dict(system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=2000,
                         response_mime_type="application/json", response_schema=SCHEMA)
    try:
        cfg = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="minimal"), **config_kwargs)
        resp = client.models.generate_content(model=MODEL_NAME, contents=prompt, config=cfg)
    except Exception as e:
        if "thinking" in str(e).lower():           # model rejects thinking_level -> retry without it
            print("(model rejected thinking_level, retrying without it)")
            resp = client.models.generate_content(model=MODEL_NAME, contents=prompt,
                                                  config=types.GenerateContentConfig(**config_kwargs))
        else:
            raise
    API_CALLS[task] += 1
    return resp.text

def analyse_note(note_text, prompt=None, task="4.1"):
    """Send ONE note to Gemini, return the parsed dict."""
    raw = _generate(prompt or build_craft_prompt(note_text), task)
    return json.loads(raw)

notes_all = pd.read_csv(NOTES_FILE)
n001 = notes_all.loc[notes_all.note_id == "N001", "note_text"].iloc[0]
result_41 = analyse_note(n001, task="4.1")
print(result_41)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'The patient missed the appointment because she did not receive a reminder and forgot the scheduled date.', 'recommended_action': "Audit the automated SMS/WhatsApp reminder system to ensure delivery to this patient's contact number."}


In [9]:
# (a) Pydantic model + allowed-list check
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_allowed(root_cause):
    if root_cause not in ALLOWED:
        raise ValueError(f"root_cause '{root_cause}' not in allowed list")

FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
rows = []
for _, r in notes_all.head(10).iterrows():                     # (b) first 10 rows
    note_text = r["note_text"]
    # input validation: NaN first, then empty / only spaces -> no API call
    if pd.isna(note_text) or str(note_text).strip() == "":
        rows.append(dict(note_id=r.note_id, appointment_id=r.appointment_id, root_cause="other",
                         confidence=0.0, summary="No note recorded", recommended_action="Re-call patient",
                         validation_status="skipped_blank", raw_response=""))
        print(r.note_id, "-> skipped_blank (no API call)")
        continue
    raw, data, status = "", None, "invalid"
    try:
        raw = _generate(build_craft_prompt(str(note_text)), "4.2")
        data = json.loads(raw)
        FollowupInsight(**data)                  # Pydantic validation
        check_allowed(data["root_cause"])        # allowed-list check
        status = "valid"
    except Exception as e:                       # one bad reply never stops the loop
        print("  problem:", str(e).splitlines()[0])
    # keep reply exactly as returned; copy fields it had, leave the rest empty (never repair)
    d = data if isinstance(data, dict) else {}
    rows.append(dict(note_id=r.note_id, appointment_id=r.appointment_id,
                     **{f: d.get(f) for f in FIELDS}, validation_status=status, raw_response=raw))
    print(r.note_id, "->", status, "|", d.get("root_cause"))
    time.sleep(13)                               # (c) free-tier rate limit

notes_analysed = pd.DataFrame(rows, columns=["note_id", "appointment_id", *FIELDS,
                                             "validation_status", "raw_response"])
notes_analysed.to_csv("notes_analysed.csv", index=False)      # (d)
print(notes_analysed.shape)
display(notes_analysed.drop(columns="raw_response"))

# agreement check against the expected key
expected = dict(N001="forgot_or_no_reminder", N002="distance_or_transport", N003="cost_or_fee",
                N005="illness_or_emergency", N006="work_or_schedule", N007="forgot_or_no_reminder",
                N008="distance_or_transport", N009="other", N010="work_or_schedule")
agree = sum(notes_analysed.set_index("note_id").root_cause.get(k) == v for k, v in expected.items())
print(f"Agreement with expected key: {agree} of 9 (need at least 7)")

N001 -> valid | forgot_or_no_reminder
N002 -> valid | distance_or_transport
N003 -> valid | cost_or_fee
N004 -> skipped_blank (no API call)
N005 -> valid | illness_or_emergency
N006 -> valid | work_or_schedule
N007 -> valid | forgot_or_no_reminder
N008 -> valid | distance_or_transport
N009 -> valid | other
N010 -> valid | work_or_schedule
(10, 8)


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status
0,N001,A0872,forgot_or_no_reminder,1.0,The patient missed the appointment because she...,Audit the automated SMS/WhatsApp reminder syst...,valid
1,N002,A0897,distance_or_transport,1.0,Patient missed the appointment due to the long...,Offer tele-consultation options for routine ch...,valid
2,N003,A0526,cost_or_fee,1.0,The patient skipped the appointment because th...,Provide information on follow-up discount poli...,valid
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank
4,N005,A0534,illness_or_emergency,1.0,The patient missed the appointment due to high...,Follow up in three days to check on recovery a...,valid
5,N006,A0507,work_or_schedule,1.0,The patient missed the appointment due to an u...,Offer extended evening hours or late-day appoi...,valid
6,N007,A0333,forgot_or_no_reminder,1.0,The patient forgot the appointment and claimed...,Verify the patient's mobile number and ensure ...,valid
7,N008,A0222,distance_or_transport,1.0,Patient missed the appointment due to a city b...,Offer tele-consultation services or provide in...,valid
8,N009,A0977,other,1.0,The patient could not be reached after three a...,Attempt to contact the patient via an alternat...,valid
9,N010,A0596,work_or_schedule,1.0,Patient missed the appointment due to a factor...,Reschedule the patient for a Saturday slot as ...,valid


Agreement with expected key: 9 of 9 (need at least 7)


In [10]:
# (a) two deliberately bad replies
bad_replies = {
    "confidence = 1.7": {"root_cause": "other", "confidence": 1.7,
                         "summary": "Patient unreachable", "recommended_action": "Re-call patient"},
    "missing recommended_action": {"root_cause": "other", "confidence": 0.5,
                                   "summary": "Patient unreachable"},
}
for name, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
        print(f"[{name}] unexpectedly passed")
    except ValidationError as e:
        print(f"[{name}] caught ValidationError:\n{e}\n")

[confidence = 1.7] caught ValidationError:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

[missing recommended_action] caught ValidationError:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...: 'Patient unreachable'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing



In [11]:
# (b) V1 vs V2 on N007 (Hinglish) and N009 (ambiguous), same schema/settings, only prompt differs
def safe_label(prompt, task):
    try:
        rc = json.loads(_generate(prompt, task)).get("root_cause")
    except Exception as e:
        rc = f"ERROR: {str(e)[:40]}"
    return rc

na = notes_analysed.set_index("note_id")
table = []
for nid in ["N007", "N009"]:
    text = notes_all.loc[notes_all.note_id == nid, "note_text"].iloc[0]
    v1 = safe_label(build_v1_prompt(text), "4.3"); time.sleep(13)           # new call
    v2 = na.loc[nid, "root_cause"]                                           # reuse 4.2 result (same CRAFT prompt)
    if pd.isna(v2):
        v2 = safe_label(build_craft_prompt(text), "4.3"); time.sleep(13)
    table.append({"note": nid, "V1 root_cause": v1, "V1 allowed?": v1 in ALLOWED,
                  "V2 root_cause": v2, "V2 allowed?": v2 in ALLOWED})
display(pd.DataFrame(table))
print("Deploy V2: it lists the allowed values and says to answer 'other' rather than guess, "
      "so replies stay inside the label set; V1 can return free text that fails the allowed-list check.")

# (c) total API calls
total = sum(API_CALLS.values())
print("API calls by task:", API_CALLS)
print("TOTAL API calls used in Tasks 4.1-4.3:", total)

,note,V1 root_cause,V1 allowed?,V2 root_cause,V2 allowed?
0,N007,Forgetfulness and lack of notification,False,forgot_or_no_reminder,True
1,N009,Unreachable,False,other,True


Deploy V2: it lists the allowed values and says to answer 'other' rather than guess, so replies stay inside the label set; V1 can return free text that fails the allowed-list check.
API calls by task: {'4.1': 1, '4.2': 9, '4.3': 2}
TOTAL API calls used in Tasks 4.1-4.3: 12


In [13]:
appts = pd.read_csv(APPT_FILE)
# light cleaning so the merge key is reliable
appts = appts.drop_duplicates().drop_duplicates(subset="appointment_id", keep="first")
appts["reminder_sent"] = appts["reminder_sent"].astype(str).str.strip().str.title()
appts.to_csv("appointments_clean.csv", index=False)

merged = notes_analysed.merge(appts, on="appointment_id", how="left")
print("merged rows:", len(merged))
display(merged[["note_id", "appointment_id", "root_cause", "reminder_sent", "clinic_id", "status"]])

ct = pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True)
display(ct)

no_rem = int((merged.reminder_sent == "No").sum())
forgot = merged[merged.root_cause == "forgot_or_no_reminder"]
print(f"Notes with reminder_sent = No: {no_rem} of {len(merged)}")
print(f"forgot_or_no_reminder notes with reminder_sent = No: {int((forgot.reminder_sent == 'No').sum())} of {len(forgot)}")

merged rows: 10


,note_id,appointment_id,root_cause,reminder_sent,clinic_id,status
0,N001,A0872,forgot_or_no_reminder,No,C05,No-Show
1,N002,A0897,distance_or_transport,No,C07,No-Show
2,N003,A0526,cost_or_fee,Yes,C04,No-Show
3,N004,A0926,other,No,C04,No-Show
4,N005,A0534,illness_or_emergency,Yes,C04,No-Show
5,N006,A0507,work_or_schedule,Yes,C04,No-Show
6,N007,A0333,forgot_or_no_reminder,No,C08,No-Show
7,N008,A0222,distance_or_transport,Yes,C02,No-Show
8,N009,A0977,other,Yes,C01,No-Show
9,N010,A0596,work_or_schedule,No,C04,No-Show


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10


Notes with reminder_sent = No: 5 of 10
forgot_or_no_reminder notes with reminder_sent = No: 2 of 2
